Now we are going to build a continuous chat loop. We want to be able to type a message, press Enter, and watch the model stream its answer back word-by-word (just like the ChatGPT interface), all while remembering what we said earlier.

In [ ]:
import ollama
import sys

# 1. Initialize the state (the memory)
    # create our messages list. Think of this as the transcript of the conversation. 
    # We start by slipping the system instructions at the very top so the model knows how to behave from the first word.
messages = [
    {'role': 'system', 'content': 'You are a helpful and concise assistant.'}
]

print("Chat started! Type 'exit' to quit.\n" + "-"*30)

# 2. Start the continuous loop
while True:
    user_input = input("You: ")
    
    print(f"User: {user_input} \n" + "-"*30)  # Debugging statement
    
    if user_input.lower() == 'exit':
        print("Goodbye!")
        break
    
    # 3. Append the user's message to the state
        # When you type a message and press Enter, the script immediately takes your text, 
        # packages it into a dictionary with the user role, and appends it to our growing transcript.
    messages.append({'role': 'user', 'content': user_input})
    
    print("Assistant: ", end="", flush=True)
    
    # 4. Stream the response
        # We send the entire transcript (messages) to Llama 3.1. But notice stream=True.
        # Normally, an LLM processes your prompt, generates the entire answer in its head, and then sends a massive block of text back all at once. 
        # This causes that annoying lag. By turning on streaming, we tell Ollama: "Send me each word (token) the exact millisecond you generate it."
    response = ollama.chat(
        model='llama3.1',
        messages=messages,
        stream=True 
    )
    
    # 5. Capture the streamed chunks and print them immediately
    full_response = ""
    # We use a for loop to catch each chunk as it arrives over the network.
    for chunk in response:
        content = chunk['message']['content']
        # print(content, end="", flush=True) slaps that tiny piece of text onto your screen immediately without dropping to a new line, creating that cool typewriter effect.
            # Because we used end="", we are explicitly telling Python not to add a newline at the end of each chunk. 
            # If we didn't use flush=True, Python would load that first chunk intfo the delivery truck and just... wait. 
            # It would wait for the next chunk, and the next, holding all the words hostage in the buffer until the 
            # AI completely finished its paragraph and we finally printed a newline.
        print(content, end="", flush=True)
        # Simultaneously, we are gluing all those chunks together into a single string called full_response behind the scenes.
        full_response += content
    
    # How does the loop know when to stop naturally?
        # End of Sequence (EOS) token from the AI
        # Ollama sees EOS token, then outputs a final chunk with a special flag like 'done': True
        # When you used stream=True, Python created a special object called a Generator.
            # When Ollama closes the network connection, the Generator essentially raises a hidden flag in Python called StopIteration
        
    print("\n" + "-"*30)
    
    # 6. Append the assistant's final complete response to the state
    messages.append({'role': 'assistant', 'content': full_response})

Chat started! Type 'exit' to quit.
------------------------------
User: hi! 
------------------------------
Assistant: Hello! How can I assist you today?
------------------------------
User: How are you today? 
------------------------------
Assistant: I'm doing well, thanks for asking! I'm a large language model, so I don't have feelings like humans do, but I'm functioning properly and ready to help with any questions or tasks you may have. How about you?
------------------------------
User: exit 
------------------------------
Goodbye!
